In [7]:
# Cell 0 - chuẩn bị (chạy 1 lần mỗi khi mở notebook mới / restart kernel)
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # US = cùng vị trí với dữ liệu

def mb(n):
    # bytes -> MB cho dễ đọc; None (không có số) thì coi là 0
    return round((n or 0) / 1024 / 1024, 2)

def stats(sql, max_mb=3000):
    """Chạy THẬT một câu SQL và in số byte processed / billed."""
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,                       # tắt cache: nếu bật, chạy lại cùng câu sẽ ra 0 byte -> đo sai
        maximum_bytes_billed=max_mb * 1024 * 1024,   # cầu chì, giống tham số max_mb của run()
    )
    job = client.query(sql, job_config=cfg)
    job.result()                                      # chờ query chạy xong
    print(f"processed = {mb(job.total_bytes_processed)} MB | billed = {mb(job.total_bytes_billed)} MB")
    return job

# Cell 0b - hàm check (dry run: chỉ ước lượng, không chạy thật, không tốn quota)
def check(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)  # dry_run=True: chỉ tính, không chạy
    job = client.query(sql, job_config=cfg)
    print(f"Ước lượng sẽ quét: {mb(job.total_bytes_processed)} MB")

In [2]:
# Cell 1 - xem bảng gốc
t = client.get_table(f"{PROJECT}.lab.fact_items_plain")
print("rows:", t.num_rows, "| size MB:", mb(t.num_bytes))
for f in t.schema:                                   # in tên cột và kiểu dữ liệu
    print(" ", f.name, f.field_type)
print("partition:", t.time_partitioning, "| cluster:", t.clustering_fields)

rows: 180771 | size MB: 14.18
  order_item_id INTEGER
  order_id INTEGER
  user_id INTEGER
  product_id INTEGER
  order_date_key INTEGER
  status STRING
  sale_price FLOAT
  cost FLOAT
  gross_profit FLOAT
  order_date DATE
partition: None | cluster: None


In [3]:
# Cell 2 - tạo bản sao có clustering
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT}.lab.fact_items_clu`
CLUSTER BY product_id          -- sắp xếp vật lý theo product_id (tối đa 4 cột, thứ tự cột có ý nghĩa)
AS
SELECT * FROM `{PROJECT}.lab.fact_items_plain`   -- SELECT * chấp nhận được vì đây là bảng nhỏ và cần copy đủ cột
"""
client.query(sql).result()
print(client.get_table(f"{PROJECT}.lab.fact_items_clu").clustering_fields)

['product_id']


In [4]:
# Cell 3 - đo bảng nhỏ
# Lấy 1 product_id bán nhiều nhất để lọc
pid = int(client.query(f"""
    SELECT product_id FROM `{PROJECT}.lab.fact_items_plain`
    GROUP BY product_id ORDER BY COUNT(*) DESC LIMIT 1
""").to_dataframe().iloc[0, 0])
print("product_id dùng để lọc:", pid)

for name in ("fact_items_plain", "fact_items_clu"):
    print(name)
    stats(f"SELECT SUM(sale_price) AS revenue FROM `{PROJECT}.lab.{name}` WHERE product_id = {pid}")

product_id dùng để lọc: 22202
fact_items_plain
processed = 2.76 MB | billed = 10.0 MB
fact_items_clu
processed = 2.76 MB | billed = 10.0 MB


In [5]:
# Cell 4 - tạo big_plain và big_clu (bảng thí nghiệm ~2,5 GB mỗi bảng)
t = client.get_table(f"{PROJECT}.lab.fact_items_plain")
REP = max(1, round(2.5 * 1024**3 / t.num_bytes))     # số lần nhân để bảng nặng ~2,5 GB
print("REP =", REP, "-> khoảng", REP * t.num_rows, "dòng mỗi bảng")

# Cùng một nguồn, chỉ khác CLUSTER BY, để so sánh công bằng
for name, cluster in (("big_plain", ""), ("big_clu", "CLUSTER BY product_id")):
    sql = f"""
    CREATE OR REPLACE TABLE `{PROJECT}.lab.{name}`
    {cluster}
    AS
    SELECT p.*, rep                                   -- rep: cột phụ cho biết đây là bản sao thứ mấy
    FROM `{PROJECT}.lab.fact_items_plain` AS p
    CROSS JOIN UNNEST(GENERATE_ARRAY(1, {REP})) AS rep -- giống SQL: nối mỗi dòng với dãy số 1..REP
    """
    client.query(sql).result()
    print(name, "xong:", mb(client.get_table(f"{PROJECT}.lab.{name}").num_bytes), "MB")

REP = 181 -> khoảng 32719551 dòng mỗi bảng
big_plain xong: 2816.2 MB
big_clu xong: 2816.2 MB


In [8]:
# Cell 5 - so sánh chi phí big_plain và big_clu
sql_plain = f"SELECT SUM(sale_price) AS revenue FROM `{PROJECT}.lab.big_plain` WHERE product_id = {pid}"
sql_clu   = f"SELECT SUM(sale_price) AS revenue FROM `{PROJECT}.lab.big_clu`   WHERE product_id = {pid}"

print("Dry run bảng clustered (chỉ là ước lượng):")
check(sql_clu)           # hàm check của bạn: chỉ in số MB ước lượng, không chạy thật

print("Chạy thật:")
print("big_plain"); stats(sql_plain)
print("big_clu");   stats(sql_clu)

Dry run bảng clustered (chỉ là ước lượng):
Ước lượng sẽ quét: 6.97 MB
Chạy thật:
big_plain
processed = 499.26 MB | billed = 500.0 MB
big_clu
processed = 6.97 MB | billed = 10.0 MB


QueryJob<project=bq-learning-510104, location=US, id=ed59aef1-ae18-4102-8569-78cc96564f29>

In [9]:
# Cell 5b - kiểm chứng: BigQuery chỉ đọc những cột được nhắc đến (dry run)
for label, cols in (("chỉ product_id", "product_id"),
                    ("product_id + sale_price", "product_id, sale_price"),
                    ("toàn bộ cột (SELECT *)", "*")):
    print(label)
    check(f"SELECT {cols} FROM `{PROJECT}.lab.big_plain`")

chỉ product_id
Ước lượng sẽ quét: 249.63 MB
product_id + sale_price
Ước lượng sẽ quét: 499.26 MB
toàn bộ cột (SELECT *)
Ước lượng sẽ quét: 2816.2 MB


In [12]:
# list_rows: đọc thẳng vài dòng đầu của bảng, KHÔNG tạo query job nên không tính byte quét
# max_results=5: chỉ lấy 5 dòng
rows = client.list_rows(f"{PROJECT}.lab.big_clu", max_results=5)

for r in rows:
    print(dict(r))       # dict(r): đổi mỗi dòng thành dạng {tên_cột: giá_trị} cho dễ đọc

{'order_item_id': 136956, 'order_id': 94436, 'user_id': 75702, 'product_id': 628, 'order_date_key': 20250125, 'status': 'Complete', 'sale_price': 3.1099998950958248, 'cost': 1.844229933064878, 'gross_profit': 1.2657699620309468, 'order_date': datetime.date(2025, 1, 25), 'rep': 6}
{'order_item_id': 142769, 'order_id': 98455, 'user_id': 78983, 'product_id': 628, 'order_date_key': 20250609, 'status': 'Shipped', 'sale_price': 3.1099998950958248, 'cost': 1.844229933064878, 'gross_profit': 1.2657699620309468, 'order_date': datetime.date(2025, 6, 9), 'rep': 6}
{'order_item_id': 102596, 'order_id': 70779, 'user_id': 56598, 'product_id': 628, 'order_date_key': 20260929, 'status': 'Returned', 'sale_price': 3.1099998950958248, 'cost': 1.844229933064878, 'gross_profit': 1.2657699620309468, 'order_date': datetime.date(2026, 9, 29), 'rep': 13}
{'order_item_id': 142769, 'order_id': 98455, 'user_id': 78983, 'product_id': 628, 'order_date_key': 20250609, 'status': 'Shipped', 'sale_price': 3.10999989509

In [16]:
# Cell 6 - các query vừa chạy (INFORMATION_SCHEMA.JOBS)
sql = rf"""
SELECT
  creation_time,
  ROUND(total_bytes_processed / 1024 / 1024, 2) AS processed_mb,
  ROUND(total_bytes_billed    / 1024 / 1024, 2) AS billed_mb,
  cache_hit,
  ARRAY_TO_STRING(ARRAY(SELECT t.table_id FROM UNNEST(referenced_tables) AS t), ', ') AS tables_used,
  LEFT(REGEXP_REPLACE(query, r'\s+', ' '), 70) AS query_head   -- gọn xuống 1 dòng, 70 ký tự đầu
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND job_type = 'QUERY' AND state = 'DONE' AND statement_type = 'SELECT'
ORDER BY creation_time DESC
LIMIT 20
"""
check(sql)
df = client.query(sql).to_dataframe()
df

Ước lượng sẽ quét: 0.12 MB


,creation_time,processed_mb,billed_mb,cache_hit,tables_used,query_head
0,2026-10-05 09:49:40.639000+00:00,0.03,20.0,False,INFORMATION_SCHEMA.JOBS,"SELECT creation_time, -- giờ UTC (giờ Việt Na..."
1,2026-10-05 09:40:04.200000+00:00,0.05,20.0,False,INFORMATION_SCHEMA.JOBS,"SELECT ROUND(total_bytes_billed / POW(1024, 3..."
2,2026-10-05 09:38:55.977000+00:00,0.00,0.0,False,INFORMATION_SCHEMA.JOBS,"SELECT creation_time, ROUND(total_bytes_proce..."
3,2026-10-05 04:33:10.354000+00:00,6.97,10.0,False,big_clu,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
4,2026-10-05 04:33:09.093000+00:00,499.26,500.0,False,big_plain,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
5,2026-10-05 04:25:17.281000+00:00,2.76,10.0,False,fact_items_clu,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
6,2026-10-05 04:25:16.128000+00:00,2.76,10.0,False,fact_items_plain,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
7,2026-10-05 04:25:12.880000+00:00,1.38,10.0,False,fact_items_plain,SELECT product_id FROM `bq-learning-510104.la...
8,2026-10-05 03:01:18.083000+00:00,0.20,10.0,False,fact_items_part,SELECT SUM(sale_price) AS tong FROM `bq-learn...
9,2026-10-05 03:00:20.329000+00:00,NaN,NaN,False,fact_items_part,SELECT SUM(sale_price) FROM `bq-learning-51010...


In [15]:
# Cell 6b - chẩn đoán vì sao Cell 6 rỗng: nới lỏng cửa sổ thời gian và bỏ lọc statement_type
from datetime import datetime, timezone

sql = rf"""
SELECT
  creation_time,                                       -- giờ UTC (giờ Việt Nam = UTC + 7)
  statement_type,                                      -- loại câu lệnh: SELECT, CREATE_TABLE_AS_SELECT...
  job_type,
  state,
  ROUND(total_bytes_processed / 1024 / 1024, 2) AS processed_mb,
  ROUND(total_bytes_billed    / 1024 / 1024, 2) AS billed_mb,
  cache_hit,
  LEFT(REGEXP_REPLACE(query, r'\s+', ' '), 60) AS query_head
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)   -- 24 giờ thay vì 3 giờ
ORDER BY creation_time DESC
LIMIT 30
"""
df6b = client.query(sql).to_dataframe()
print("Bây giờ (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"))
df6b

Bây giờ (UTC): 2026-10-05 09:49


,creation_time,statement_type,job_type,state,processed_mb,billed_mb,cache_hit,query_head
0,2026-10-05 09:40:04.200000+00:00,SELECT,QUERY,DONE,0.05,20.0,False,"SELECT ROUND(total_bytes_billed / POW(1024, 3..."
1,2026-10-05 09:38:55.977000+00:00,SELECT,QUERY,DONE,0.00,0.0,False,"SELECT creation_time, ROUND(total_bytes_proce..."
2,2026-10-05 04:33:10.354000+00:00,SELECT,QUERY,DONE,6.97,10.0,False,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
3,2026-10-05 04:33:09.093000+00:00,SELECT,QUERY,DONE,499.26,500.0,False,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
4,2026-10-05 04:27:22.364000+00:00,CREATE_TABLE_AS_SELECT,QUERY,DONE,14.18,15.0,False,CREATE OR REPLACE TABLE `bq-learning-510104.l...
5,2026-10-05 04:27:02.915000+00:00,CREATE_TABLE_AS_SELECT,QUERY,DONE,14.18,15.0,False,CREATE OR REPLACE TABLE `bq-learning-510104.l...
6,2026-10-05 04:25:17.281000+00:00,SELECT,QUERY,DONE,2.76,10.0,False,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
7,2026-10-05 04:25:16.128000+00:00,SELECT,QUERY,DONE,2.76,10.0,False,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
8,2026-10-05 04:25:12.880000+00:00,SELECT,QUERY,DONE,1.38,10.0,False,SELECT product_id FROM `bq-learning-510104.la...
9,2026-10-05 04:23:22.537000+00:00,CREATE_TABLE_AS_SELECT,QUERY,DONE,14.18,15.0,False,CREATE OR REPLACE TABLE `bq-learning-510104.l...


In [14]:
# Cell 7 - 5 query tốn nhất trong 7 ngày
sql = rf"""
SELECT
  ROUND(total_bytes_billed / POW(1024, 3), 3) AS billed_gb,
  ROUND(total_bytes_billed / POW(1024, 4) * 6.25, 4) AS est_usd,  -- ước tính nếu tính tiền: 6,25 USD/TiB
  LEFT(REGEXP_REPLACE(query, r'\s+', ' '), 70) AS query_head
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND job_type = 'QUERY' AND state = 'DONE' AND total_bytes_billed IS NOT NULL
ORDER BY total_bytes_billed DESC
LIMIT 5
"""
client.query(sql).to_dataframe()

,billed_gb,est_usd,query_head
0,0.488,0.0030,SELECT SUM(sale_price) AS revenue FROM `bq-lea...
1,0.029,0.0002,"select d.year ,d.quarter ,p.category ,sum(i.s..."
2,0.029,0.0002,"select count(*) as amount ,count(distinct(ord..."
3,0.029,0.0002,"select d.year ,d.quarter ,p.category ,sum(i.s..."
4,0.020,0.0001,"SELECT MIN(d.full_date) AS min_d, -- ngày sớm..."


In [17]:
# Cell 8 - dọn dẹp: chỉ xóa 2 bảng thí nghiệm big_*
for name in ("big_plain", "big_clu"):
    client.delete_table(f"{PROJECT}.lab.{name}", not_found_ok=True)
print([t.table_id for t in client.list_tables(f"{PROJECT}.lab")])

['fact_items_clu', 'fact_items_part', 'fact_items_plain', 'fact_items_recent']
